# P04 — Perubahan Mangrove Pesisir Merauke 2005–2025 dan Hubungannya dengan Abrasi

| Tahap | Isi | Waktu |
|---|---|---|
| 0–2 | Instalasi, login Earth Engine, Google Drive, konfigurasi | 2 menit |
| 3 | Komposit Landsat 5/7/8/9 untuk 5 epoch (2005, 2010, 2015, 2020, 2025), **disimpan di Drive** | 15–30 menit |
| 4 | Peta acuan mangrove: Global Mangrove Watch v3, CGMD-Extent30, ESA WorldCover, DEM Copernicus | 5 menit |
| 5 | Label otomatis dari **kesepakatan peta acuan** (mangrove stabil vs bukan mangrove stabil), Random Forest, validasi silang spasial dan antar-epoch | 5–10 menit |
| 6 | Peta mangrove 5 epoch, luas, kesesuaian dengan peta acuan | 5–10 menit |
| 7 | Perubahan 2005–2025: hilang ke air (abrasi) vs berubah menjadi daratan lain; bertambah | 1 menit |
| 8 | **Hubungan dengan abrasi:** lebar mangrove per transek P01 vs laju garis vegetasi (LRR) | 2 menit |
| 9 | Gambar dua bahasa dan unduh `P04_hasil.zip` | 2 menit |

**Tanpa survei lapangan dan tanpa penilaian manual.** Data P01 (transek dan laju abrasi) dibaca otomatis dari Google Drive `P01_abrasi/P01_hasil.zip`; bila tidak ada, dari Zenodo (DOI 10.5281/zenodo.24421128) atau GitHub.

**Cara menjalankan:** *Runtime → Jalankan semua*. Izinkan login Earth Engine dan Google Drive di awal. Bila Colab terputus, jalankan lagi; epoch yang sudah tersimpan di Drive tidak diunduh ulang.

In [ ]:
# 0. Instalasi
!pip install -q earthengine-api scikit-learn pandas matplotlib scipy openpyxl requests

In [ ]:
# 1. Earth Engine + Google Drive
import os
GEE_PROJECT = 'riset-psn-pps'
try:
    import ee
    try:
        ee.Initialize(project=GEE_PROJECT)
    except Exception:
        ee.Authenticate(force=True); ee.Initialize(project=GEE_PROJECT)
    print('Earth Engine siap:', ee.String('OK').getInfo())
except Exception as e:
    print('Earth Engine tidak tersedia:', str(e)[:120])
try:
    from google.colab import drive; drive.mount('/content/drive'); DRIVE = '/content/drive/MyDrive'
except Exception:
    DRIVE = '.'
BASE = os.path.join(DRIVE, 'P04_mangrove'); os.makedirs(BASE, exist_ok=True); print('Folder kerja:', BASE)

In [ ]:
# 2. Konfigurasi
import json, time, zipfile, math, io, glob
import numpy as np, pandas as pd
TOPIC = 'P04'; SEED = 42
BBOX = [140.15, -8.80, 140.75, -8.30]      # sama dengan P01 (pantai Kota Merauke dan sekitarnya)
EPOCHS = [2005, 2010, 2015, 2020, 2025]
WIN = {2005: (2004, 2006), 2010: (2009, 2011), 2015: (2014, 2016), 2020: (2019, 2021), 2025: (2024, 2025)}   # tahun awal-akhir komposit
EP_LAB = [2005, 2010, 2015, 2020]           # epoch yang tercakup peta acuan (dipakai untuk melatih dan menguji)
RES, CRS = 30, 'EPSG:32754'
BANDS = ['B', 'G', 'R', 'N', 'S1', 'S2', 'NDVI_p10', 'NDVI_p50', 'NDVI_p90', 'MNDWI']
FEAT = BANDS + ['MVI', 'DEM']
GMW_YEARS = [2007, 2010, 2015, 2020]; CGMD_YEARS = [2005, 2010, 2015, 2020, 2023]
N_MAN, N_LAND, N_WAT = 3000, 3000, 1000     # titik latih per kelas (lokasi sama untuk semua epoch)
BLOCK_M = 2000                              # ukuran blok spasial validasi silang (m)
DEM_MAX = 20                                # zona analisis: elevasi <= 20 m
STEP, L_LAND, L_SEA = 5, 1000, 1500         # geometri transek P01 (m)
FRONT_M = 150                               # zona depan: 150 m ke darat dari garis vegetasi 2005
BOOT, BLK_T = 2000, 20                      # bootstrap blok: 2000 ulangan, blok 20 transek (2 km)
P01_DOI_RECORD = '24421128'; P01_REPO = 'marsujitullah-droid/merauke-pantai-P01-abrasi'
CACHE = os.path.join(BASE, 'cache'); os.makedirs(CACHE, exist_ok=True)
OUT = os.path.join(BASE, 'P04_hasil'); DIR_TAB, DIR_FIG, DIR_DATA = [os.path.join(OUT, d) for d in ('03_tabel', '02_gambar', '07_dataset')]
for d_ in (DIR_TAB, DIR_FIG, DIR_DATA): os.makedirs(d_, exist_ok=True)
COLS_EN = {'epoch': 'epoch', 'tahun_awal': 'start_year', 'tahun_akhir': 'end_year', 'jumlah_citra': 'n_images', 'piksel_valid_persen': 'valid_pixels_pct',
           'skema': 'scheme', 'epoch_uji': 'test_epoch', 'akurasi': 'overall_accuracy', 'F1_mangrove': 'F1_mangrove', 'kappa': 'kappa', 'n_uji': 'n_test',
           'fitur': 'feature', 'kepentingan': 'importance', 'sumber': 'source', 'tahun': 'year', 'luas_ha': 'area_ha', 'peta_acuan': 'reference_map',
           'epoch_peta': 'map_epoch', 'presisi': 'precision', 'recall': 'recall', 'periode': 'period', 'hilang_ha': 'loss_ha', 'bertambah_ha': 'gain_ha',
           'bersih_ha': 'net_ha', 'laju_bersih_persen_per_thn': 'net_rate_pct_per_yr', 'kelas': 'class', 'n_piksel_acuan': 'n_reference_pixels',
           'terdeteksi_persen': 'detected_pct', 'jenis': 'type', 'persen': 'percent', 'tipe_depan': 'front_type', 'n_transek': 'n_transects',
           'median_LRR_veg': 'median_LRR_veg', 'abrasi_signifikan_persen': 'significant_erosion_pct', 'median_lebar_2005_m': 'median_width_2005_m',
           'median_lebar_2025_m': 'median_width_2025_m', 'uji': 'test', 'statistik': 'statistic', 'nilai_p': 'p_value', 'CI95_bawah': 'CI95_low', 'CI95_atas': 'CI95_high',
           'variabel_x': 'x_variable', 'variabel_y': 'y_variable', 'rho_Spearman': 'Spearman_rho', 'n': 'n', 'besaran': 'quantity', 'nilai': 'value',
           'transek': 'transect', 'jarak_km': 'alongshore_km', 'lebar_2005_m': 'width_2005_m', 'lebar_2025_m': 'width_2025_m', 'delta_lebar_m': 'width_change_m',
           'fraksi_depan_mangrove': 'front_mangrove_fraction', 'posisi_veg_2005_m': 'vegline_2005_m'}
VAL_EN = {'Spasial (5 lipatan)': 'Spatial (5 folds)', 'Spasial + antar-epoch': 'Spatial + cross-epoch', 'Peta ini': 'This map', 'Mangrove stabil': 'Stable mangrove',
          'Hilang ke air': 'Lost to water', 'Hilang ke daratan lain': 'Lost to other land', 'Bertambah dari air': 'Gained from water', 'Bertambah dari daratan': 'Gained from land',
          'Bukan mangrove stabil': 'Stable non-mangrove', 'Hilang (acuan sepakat)': 'Loss (references agree)', 'Bertambah (acuan sepakat)': 'Gain (references agree)',
          'Mangrove stabil (acuan sepakat)': 'Stable mangrove (references agree)', 'Bukan mangrove stabil (acuan sepakat)': 'Stable non-mangrove (references agree)',
          'Depan mangrove': 'Mangrove-fronted', 'Depan bukan mangrove': 'Non-mangrove-fronted', 'Semua': 'All', 'Selisih median LRR_veg (mangrove − bukan)': 'Median LRR_veg difference (mangrove − non)',
          'Cliff delta': "Cliff's delta", 'Mann-Whitney U': 'Mann-Whitney U', 'ya': 'yes', 'tidak': 'no'}
def save_tab(df, name_id, name_en):
    d = df.copy()
    for c in d.columns:
        if not pd.api.types.is_numeric_dtype(d[c]): d[c] = d[c].map(lambda v: VAL_EN.get(v, v))
    df.to_csv(os.path.join(DIR_TAB, f'P04_tabel_{name_id}_ID.csv'), index=False)
    d.rename(columns=COLS_EN).to_csv(os.path.join(DIR_TAB, f'P04_table_{name_en}_EN.csv'), index=False)
def utm_to_ll(x, y, zone=54, south=True):
    # UTM -> lon/lat (WGS84), rumus standar Snyder (1987)
    a, f, k0 = 6378137.0, 1 / 298.257223563, 0.9996; e2 = f * (2 - f); ep2 = e2 / (1 - e2)
    x = np.asarray(x, float) - 500000.0; y = np.asarray(y, float) - (10000000.0 if south else 0)
    M = y / k0; mu = M / (a * (1 - e2 / 4 - 3 * e2 ** 2 / 64 - 5 * e2 ** 3 / 256)); e1 = (1 - np.sqrt(1 - e2)) / (1 + np.sqrt(1 - e2))
    p1 = mu + (3 * e1 / 2 - 27 * e1 ** 3 / 32) * np.sin(2 * mu) + (21 * e1 ** 2 / 16 - 55 * e1 ** 4 / 32) * np.sin(4 * mu) + (151 * e1 ** 3 / 96) * np.sin(6 * mu)
    N1 = a / np.sqrt(1 - e2 * np.sin(p1) ** 2); T1 = np.tan(p1) ** 2; C1 = ep2 * np.cos(p1) ** 2; R1 = a * (1 - e2) / (1 - e2 * np.sin(p1) ** 2) ** 1.5; D = x / (N1 * k0)
    lat = p1 - (N1 * np.tan(p1) / R1) * (D ** 2 / 2 - (5 + 3 * T1 + 10 * C1 - 4 * C1 ** 2 - 9 * ep2) * D ** 4 / 24 + (61 + 90 * T1 + 298 * C1 + 45 * T1 ** 2 - 252 * ep2 - 3 * C1 ** 2) * D ** 6 / 720)
    lon = (D - (1 + 2 * T1 + C1) * D ** 3 / 6 + (5 - 2 * C1 + 28 * T1 - 3 * C1 ** 2 + 8 * ep2 + 24 * T1 ** 2) * D ** 5 / 120) / np.cos(p1)
    return np.degrees(lon) + (zone - 1) * 6 - 180 + 3, np.degrees(lat)
def ll_to_utm(lon, lat, zone=54, south=True):
    # lon/lat (WGS84) -> UTM, rumus standar Snyder (1987)
    a, f, k0 = 6378137.0, 1 / 298.257223563, 0.9996; e2 = f * (2 - f); ep2 = e2 / (1 - e2)
    p = np.radians(np.asarray(lat, float)); l = np.radians(np.asarray(lon, float) - ((zone - 1) * 6 - 180 + 3))
    N = a / np.sqrt(1 - e2 * np.sin(p) ** 2); T = np.tan(p) ** 2; C = ep2 * np.cos(p) ** 2; A = np.cos(p) * l
    M = a * ((1 - e2 / 4 - 3 * e2 ** 2 / 64 - 5 * e2 ** 3 / 256) * p - (3 * e2 / 8 + 3 * e2 ** 2 / 32 + 45 * e2 ** 3 / 1024) * np.sin(2 * p)
             + (15 * e2 ** 2 / 256 + 45 * e2 ** 3 / 1024) * np.sin(4 * p) - (35 * e2 ** 3 / 3072) * np.sin(6 * p))
    x = k0 * N * (A + (1 - T + C) * A ** 3 / 6 + (5 - 18 * T + T ** 2 + 72 * C - 58 * ep2) * A ** 5 / 120) + 500000.0
    y = k0 * (M + N * np.tan(p) * (A ** 2 / 2 + (5 - T + 9 * C + 4 * C ** 2) * A ** 4 / 24 + (61 - 58 * T + T ** 2 + 600 * C - 330 * ep2) * A ** 6 / 720))
    return x, y + (10000000.0 if south else 0)
np.random.seed(SEED); print('Folder hasil:', OUT)

## Tahap 3 — Komposit Landsat per epoch
Untuk setiap epoch: semua citra Landsat Collection 2 Level-2 (Landsat 5, 7, 8, 9) dalam jendela 3 tahun (2025: 2024–2025), awan dan bayangan dibuang (QA_PIXEL bit 1, 3, 4), lalu diambil **median** enam band reflektan, **persentil 10/50/90 NDVI** (menangkap pengaruh genangan pasang), dan median MNDWI. Disimpan per epoch di Drive.

In [ ]:
F_GRID = os.path.join(CACHE, 'grid.json')
if os.path.exists(F_GRID):
    g_ = json.load(open(F_GRID)); X0, Y0, W, H = g_['X0'], g_['Y0'], g_['W'], g_['H']
else:
    rect = ee.Geometry.Rectangle(BBOX)
    ring = rect.transform(CRS, 1).bounds(1, CRS).coordinates().getInfo()[0]
    xs, ys = [p[0] for p in ring], [p[1] for p in ring]
    X0, Y0 = math.floor(min(xs) / RES) * RES, math.ceil(max(ys) / RES) * RES
    W, H = int(math.ceil((max(xs) - X0) / RES)), int(math.ceil((Y0 - min(ys)) / RES))
    json.dump({'X0': X0, 'Y0': Y0, 'W': W, 'H': H}, open(F_GRID, 'w'))
GRID = {'dimensions': {'width': W, 'height': H}, 'affineTransform': {'scaleX': RES, 'shearX': 0, 'translateX': X0, 'shearY': 0, 'scaleY': -RES, 'translateY': Y0}, 'crsCode': CRS}
print(f'Grid {W} x {H} piksel ({W * RES / 1000:.0f} x {H * RES / 1000:.0f} km), sama dengan P01')

def fetch(img):
    # batas 48 MB per permintaan: unduh per pita
    out = {}
    for b in img.bandNames().getInfo():
        out[b] = ee.data.computePixels({'expression': img.select(b), 'fileFormat': 'NUMPY_NDARRAY', 'grid': GRID})[b]
    return out
def prep(col, bands):
    def f(i):
        qa = i.select('QA_PIXEL'); m = qa.bitwiseAnd(1 << 1).eq(0).And(qa.bitwiseAnd(1 << 3).eq(0)).And(qa.bitwiseAnd(1 << 4).eq(0))
        return i.select(bands).multiply(0.0000275).add(-0.2).rename(['B', 'G', 'R', 'N', 'S1', 'S2']).updateMask(m)
    return col.filterBounds(ee.Geometry.Rectangle(BBOX)).map(f)
def epoch_img(e):
    a, b = f'{WIN[e][0]}-01-01', f'{WIN[e][1] + 1}-01-01'
    old = ['SR_B1', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B7']; new = ['SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7']
    col = (prep(ee.ImageCollection('LANDSAT/LT05/C02/T1_L2').filterDate(a, b), old).merge(prep(ee.ImageCollection('LANDSAT/LE07/C02/T1_L2').filterDate(a, b), old))
           .merge(prep(ee.ImageCollection('LANDSAT/LC08/C02/T1_L2').filterDate(a, b), new)).merge(prep(ee.ImageCollection('LANDSAT/LC09/C02/T1_L2').filterDate(a, b), new)))
    med = col.median()
    ndvi = col.map(lambda i: i.normalizedDifference(['N', 'R']).rename('NDVI')).reduce(ee.Reducer.percentile([10, 50, 90]))
    mndwi = col.map(lambda i: i.normalizedDifference(['G', 'S1']).rename('MNDWI')).median().rename('MNDWI')
    img = ee.Image.cat([med, ndvi, mndwi]).select(BANDS).multiply(10000).unmask(-32768).toInt16()
    return img, col.size()
N_IMG = {}
for e in EPOCHS:
    fn = os.path.join(CACHE, f'L_{e}.npz')
    if os.path.exists(fn):
        N_IMG[e] = int(np.load(fn)['n']); continue
    t0 = time.perf_counter()
    for attempt in range(4):
        try:
            img, n = epoch_img(e); arr = fetch(img); n = n.getInfo(); np.savez_compressed(fn, n=n, **arr); N_IMG[e] = n; break
        except Exception as ex:
            print(f'  {e}: percobaan {attempt + 1} gagal ({str(ex)[:90]}); coba lagi...'); time.sleep(20)
    print(f'{e}: {N_IMG.get(e, "GAGAL")} citra ({time.perf_counter() - t0:.0f} dtk)')
assert all(e in N_IMG for e in EPOCHS), 'Ada epoch yang gagal diunduh. Jalankan ulang sel ini.'

## Tahap 4 — Peta acuan mangrove dan DEM
* **Global Mangrove Watch v3** (Bunting et al., 2022): 2007, 2010, 2015, 2020 (katalog komunitas GEE).
* **CGMD-Extent30** (Zhang et al., 2026): mangrove tahunan 1984–2023; dipakai 2005, 2010, 2015, 2020, 2023.
* **ESA WorldCover 2021** kelas 95 (mangrove), hanya untuk pembanding.
* **Copernicus DEM GLO-30** untuk zona pesisir (≤ 20 m) dan sebagai fitur.

Peta acuan **tidak** dianggap kebenaran lapangan: hanya piksel yang **semua** peta acuannya sepakat yang dipakai sebagai label.

In [ ]:
F_REF = os.path.join(CACHE, 'acuan.npz')
if not os.path.exists(F_REF):
    rect = ee.Geometry.Rectangle(BBOX); refs = {}
    def paint(fc): return ee.Image(0).byte().paint(fc.filterBounds(rect), 1).rename('m')
    for y in GMW_YEARS:
        try:
            refs[f'GMW_{y}'] = fetch(paint(ee.FeatureCollection(f'projects/earthengine-legacy/assets/projects/sat-io/open-datasets/GMW/extent/gmw_v3_{y}_vec')))['m'].astype(np.uint8)
        except Exception as ex: print(f'GMW {y} tidak tersedia: {str(ex)[:100]}')
    cg = ee.FeatureCollection('projects/mangrovedatahub2/assets/CGMD-Extent30')
    for y in CGMD_YEARS:
        try:
            refs[f'CGMD_{y}'] = fetch(paint(cg.filter(ee.Filter.eq('year', y))))['m'].astype(np.uint8)
        except Exception as ex: print(f'CGMD {y} tidak tersedia: {str(ex)[:100]}')
    try:
        wc = ee.ImageCollection('ESA/WorldCover/v200').first().eq(95).reduceResolution(ee.Reducer.mean(), maxPixels=64).reproject(crs=CRS, scale=RES).gt(0.5).toByte().rename('m')
        refs['WorldCover_2021'] = fetch(wc)['m'].astype(np.uint8)
    except Exception as ex: print('WorldCover tidak tersedia:', str(ex)[:100])
    dem = ee.ImageCollection('COPERNICUS/DEM/GLO30').filterBounds(rect).select('DEM').mosaic().unmask(-9999).toInt16().rename('DEM')
    refs['DEM'] = fetch(dem)['DEM']
    np.savez_compressed(F_REF, **refs)
REF = dict(np.load(F_REF)); DEM = REF.pop('DEM').astype(np.float32); DEM[DEM <= -9999] = np.nan
print('Peta acuan:', {k: int(v.sum()) * RES * RES // 10000 for k, v in REF.items()}, '(ha)')
assert any(k.startswith('GMW') for k in REF) and any(k.startswith('CGMD') for k in REF), 'GMW atau CGMD tidak terunduh. Jalankan ulang sel ini.'

## Tahap 5 — Label dari kesepakatan peta acuan dan Random Forest
* **Mangrove stabil:** mangrove menurut *semua* peta GMW dan CGMD (2005–2023), dikikis 1 piksel agar piksel tepi campuran tidak ikut.
* **Bukan mangrove stabil:** bukan mangrove menurut semua peta dan berjarak > 90 m dari mangrove mana pun; dibagi menjadi **daratan** dan **air** (MNDWI median > ambang Otsu).
* Lokasi titik sama untuk keempat epoch berlabel (2005–2020), fitur diambil dari komposit masing-masing epoch.
* Validasi: (a) 5 lipatan **spasial** (blok 2 km); (b) **spasial + antar-epoch**: latih pada 3 epoch, uji pada epoch keempat di blok yang tidak dipakai melatih.

In [ ]:
from scipy import ndimage as ndi
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score, precision_score, recall_score
def load(e):
    z = np.load(os.path.join(CACHE, f'L_{e}.npz')); out = {}
    for k in BANDS:
        a = z[k].astype(np.float32); a[a <= -32768] = np.nan; out[k] = a / 10000.0
    den = np.abs(out['S1'] - out['G']); out['MVI'] = np.clip(np.where(den > 1e-3, np.abs(out['N'] - out['G']) / np.maximum(den, 1e-3), np.nan), 0, 20)   # MVI = |NIR−G| / |SWIR1−G| (Baloloy et al., 2020)
    out['DEM'] = DEM; return out
STK = {e: load(e) for e in EPOCHS}
def otsu(v, lo, hi):
    v = v[np.isfinite(v)]; v = v[(v > -1) & (v < 1)]
    if len(v) < 100: return 0.0
    h, ed = np.histogram(v, 256); c = (ed[:-1] + ed[1:]) / 2; w = np.cumsum(h); m = np.cumsum(h * c); wt, mt = w[-1], m[-1]
    b = (mt * w - m * wt) ** 2 / np.maximum(w * (wt - w), 1e-9); return float(np.clip(c[np.nanargmax(b)], lo, hi))
MN_MED = np.nanmedian(np.stack([STK[e]['MNDWI'] for e in EPOCHS]), axis=0)
THR_W = otsu(MN_MED, -0.3, 0.3); WATER = MN_MED > THR_W
VALID = {e: np.isfinite(np.stack([STK[e][k] for k in BANDS])).all(0) for e in EPOCHS}
ZONE = (np.nan_to_num(DEM, nan=0) <= DEM_MAX)
KM = [k for k in REF if k.startswith(('GMW', 'CGMD'))]
allM = np.all([REF[k] == 1 for k in KM], axis=0); anyM = np.any([REF[k] == 1 for k in KM], axis=0)
st_M = ndi.binary_erosion(allM, np.ones((3, 3))) & ZONE
st_N = ~ndi.binary_dilation(anyM, iterations=3) & ZONE
vall = np.all([VALID[e] for e in EP_LAB], axis=0)
rng = np.random.default_rng(SEED)
def pick(mask, n):
    idx = np.flatnonzero(mask & vall); return rng.choice(idx, size=min(n, len(idx)), replace=False)
I_M, I_L, I_W = pick(st_M, N_MAN), pick(st_N & ~WATER, N_LAND), pick(st_N & WATER, N_WAT)
IDX = np.r_[I_M, I_L, I_W]; Y = np.r_[np.ones(len(I_M), int), np.zeros(len(I_L) + len(I_W), int)]
SUB = np.r_[np.full(len(I_M), 'mangrove'), np.full(len(I_L), 'daratan'), np.full(len(I_W), 'air')]
rr, cc = np.unravel_index(IDX, (H, W)); GRP = (rr * RES // BLOCK_M) * 10000 + (cc * RES // BLOCK_M)
print(f'Ambang air (Otsu MNDWI) = {THR_W:.3f}; titik latih: mangrove {len(I_M)}, daratan {len(I_L)}, air {len(I_W)}; {len(np.unique(GRP))} blok spasial')
rows = []
for e in EP_LAB:
    d = pd.DataFrame({k: STK[e][k].ravel()[IDX] for k in FEAT}); d['epoch'] = e; d['y'] = Y; d['sub'] = SUB; d['grp'] = GRP; d['idx'] = IDX; rows.append(d)
TRAIN = pd.concat(rows, ignore_index=True).dropna(subset=FEAT)
lon_, lat_ = utm_to_ll(X0 + (cc + 0.5) * RES, Y0 - (rr + 0.5) * RES)
pts = pd.DataFrame({'idx': IDX, 'kelas': SUB, 'grp': GRP, 'lon': lon_, 'lat': lat_}); pts.round(6).to_csv(os.path.join(DIR_DATA, 'P04_titik_label.csv'), index=False)
TRAIN.round(5).to_csv(os.path.join(DIR_DATA, 'P04_sampel_latih.csv'), index=False)
def rf(): return RandomForestClassifier(n_estimators=200, min_samples_leaf=2, class_weight='balanced', n_jobs=-1, random_state=SEED)
def mets(y, p): return dict(akurasi=accuracy_score(y, p), F1_mangrove=f1_score(y, p), kappa=cohen_kappa_score(y, p), n_uji=len(y))
cv = []
ug = np.unique(TRAIN.grp); perm = rng.permutation(ug); fold_of = {g: i % 5 for i, g in enumerate(perm)}; TRAIN['fold'] = TRAIN.grp.map(fold_of)
P = np.zeros(len(TRAIN), int)
for k in range(5):
    tr, te = TRAIN.fold != k, TRAIN.fold == k; m = rf().fit(TRAIN.loc[tr, FEAT], TRAIN.y[tr]); P[te.values] = m.predict(TRAIN.loc[te, FEAT])
cv.append({'skema': 'Spasial (5 lipatan)', 'epoch_uji': 'Semua', **mets(TRAIN.y, P)})
for e in EP_LAB:
    yy, pp = [], []
    for k in range(5):
        tr = (TRAIN.epoch != e) & (TRAIN.fold != k); te = (TRAIN.epoch == e) & (TRAIN.fold == k)
        if te.sum() == 0: continue
        m = rf().fit(TRAIN.loc[tr, FEAT], TRAIN.y[tr]); yy.append(TRAIN.y[te].values); pp.append(m.predict(TRAIN.loc[te, FEAT]))
    cv.append({'skema': 'Spasial + antar-epoch', 'epoch_uji': str(e), **mets(np.concatenate(yy), np.concatenate(pp))})
CV = pd.DataFrame(cv).round(4); save_tab(CV, 'validasi_silang', 'cross_validation'); print(CV.to_string(index=False))
MODEL = rf().fit(TRAIN[FEAT].values, TRAIN.y.values)
IMP = pd.DataFrame({'fitur': FEAT, 'kepentingan': MODEL.feature_importances_}).sort_values('kepentingan', ascending=False).round(4)
save_tab(IMP, 'kepentingan_fitur', 'feature_importance')
DATA = pd.DataFrame([{'epoch': e, 'tahun_awal': WIN[e][0], 'tahun_akhir': WIN[e][1], 'jumlah_citra': N_IMG[e], 'piksel_valid_persen': round(100 * VALID[e][ZONE].mean(), 1)} for e in EPOCHS])
save_tab(DATA, 'data_per_epoch', 'data_per_epoch'); print(DATA.to_string(index=False))

## Tahap 6 — Peta mangrove 5 epoch, luas, dan kesesuaian dengan peta acuan
Model akhir dilatih dengan semua titik dari 4 epoch berlabel, lalu diterapkan pada kelima epoch (termasuk 2025 yang belum dicakup peta acuan). Filter mayoritas 3 × 3 mengurangi piksel acak. Kesesuaian dengan peta acuan dihitung pada **semua** piksel zona pesisir (bukan hanya piksel stabil), sehingga mencerminkan kesesuaian peta, bukan akurasi terhadap lapangan.

In [ ]:
MAP = {}
for e in EPOCHS:
    msk = ZONE & VALID[e]; ii = np.flatnonzero(msk); pr = np.zeros(len(ii), np.float32)
    X = np.stack([STK[e][k].ravel()[ii] for k in FEAT], 1)
    for s in range(0, len(ii), 400000): pr[s:s + 400000] = MODEL.predict_proba(X[s:s + 400000])[:, 1]
    m = np.zeros(H * W, np.float32); m[ii] = pr; m = (m.reshape(H, W) >= 0.5).astype(np.uint8)
    m = (ndi.uniform_filter(m.astype(np.float32), 3) > 0.5).astype(np.float32); m[~msk] = np.nan; MAP[e] = m
    print(f'{e}: mangrove {np.nansum(m) * RES * RES / 1e4:,.0f} ha')
np.savez_compressed(os.path.join(DIR_DATA, 'P04_peta_mangrove.npz'), X0=X0, Y0=Y0, RES=RES, crs=CRS, **{f'm{e}': np.nan_to_num(MAP[e], nan=255).astype(np.uint8) for e in EPOCHS})
ha = RES * RES / 1e4
AREA = [{'sumber': 'Peta ini', 'tahun': e, 'luas_ha': round(float(np.nansum(MAP[e])) * ha, 1)} for e in EPOCHS]
for k in REF: AREA.append({'sumber': k.split('_')[0], 'tahun': int(k.split('_')[1]), 'luas_ha': round(float((REF[k].astype(bool) & ZONE).sum()) * ha, 1)})
AREA = pd.DataFrame(AREA); save_tab(AREA, 'luas_mangrove', 'mangrove_area'); print(AREA.to_string(index=False))
PAIRS = [('CGMD_2005', 2005), ('GMW_2007', 2005), ('GMW_2010', 2010), ('CGMD_2010', 2010), ('GMW_2015', 2015), ('CGMD_2015', 2015), ('GMW_2020', 2020), ('CGMD_2020', 2020),
         ('WorldCover_2021', 2020), ('CGMD_2023', 2025), ('WorldCover_2021', 2025)]
AGR = []
for k, e in PAIRS:
    if k not in REF: continue
    ok = np.isfinite(MAP[e]); r_ = REF[k][ok].astype(int); p_ = MAP[e][ok].astype(int)
    AGR.append({'peta_acuan': k.replace('_', ' '), 'epoch_peta': e, 'presisi': precision_score(r_, p_, zero_division=0), 'recall': recall_score(r_, p_, zero_division=0), 'F1_mangrove': f1_score(r_, p_, zero_division=0)})
# kesesuaian antar peta acuan sebagai pembanding
for a, b in [('GMW_2010', 'CGMD_2010'), ('GMW_2015', 'CGMD_2015'), ('GMW_2020', 'CGMD_2020'), ('WorldCover_2021', 'CGMD_2020')]:
    if a in REF and b in REF:
        ok = ZONE; AGR.append({'peta_acuan': b.replace('_', ' '), 'epoch_peta': a.replace('_', ' '), 'presisi': precision_score(REF[b][ok], REF[a][ok], zero_division=0),
                               'recall': recall_score(REF[b][ok], REF[a][ok], zero_division=0), 'F1_mangrove': f1_score(REF[b][ok], REF[a][ok], zero_division=0)})
AGR = pd.DataFrame(AGR).round(4); save_tab(AGR, 'kesesuaian_peta_acuan', 'agreement_reference_maps'); print(AGR.to_string(index=False))

## Tahap 7 — Perubahan mangrove 2005–2025 dan penyebab langsungnya
* Per periode 5 tahun: luas hilang, bertambah, bersih, dan laju bersih (%/tahun).
* 2005 → 2025: **hilang ke air** (piksel menjadi air pada 2025: abrasi atau genangan) vs **hilang ke daratan lain** (pembukaan, tambak, permukiman, penggalian); **bertambah dari air** (kolonisasi dataran lumpur) vs **dari daratan**.
* Pemeriksaan perubahan: pada piksel yang perubahannya **disepakati GMW dan CGMD** 2010 → 2020, berapa persen terdeteksi peta ini.

In [ ]:
def thr_e(e): return otsu(STK[e]['MNDWI'], -0.3, 0.3)
PER = []
for a, b in zip(EPOCHS[:-1], EPOCHS[1:]):
    ok = np.isfinite(MAP[a]) & np.isfinite(MAP[b]); A, B = MAP[a][ok] == 1, MAP[b][ok] == 1
    loss, gain = (A & ~B).sum() * ha, (~A & B).sum() * ha; base = A.sum() * ha
    PER.append({'periode': f'{a}–{b}', 'hilang_ha': round(loss, 1), 'bertambah_ha': round(gain, 1), 'bersih_ha': round(gain - loss, 1), 'laju_bersih_persen_per_thn': round(100 * (gain - loss) / max(base, 1e-9) / (b - a), 2)})
a, b = EPOCHS[0], EPOCHS[-1]; ok = np.isfinite(MAP[a]) & np.isfinite(MAP[b]); A, B = MAP[a] == 1, MAP[b] == 1
loss, gain = (A & ~B).sum() * ha, (~A & B).sum() * ha
PER.append({'periode': f'{a}–{b}', 'hilang_ha': round(loss, 1), 'bertambah_ha': round(gain, 1), 'bersih_ha': round(gain - loss, 1), 'laju_bersih_persen_per_thn': round(100 * (gain - loss) / max(A[ok].sum() * ha, 1e-9) / (b - a), 2)})
PER = pd.DataFrame(PER); save_tab(PER, 'perubahan_per_periode', 'change_per_period'); print(PER.to_string(index=False))
W_A, W_B = STK[a]['MNDWI'] > thr_e(a), STK[b]['MNDWI'] > thr_e(b)
CH = np.full((H, W), np.nan, np.float32)            # 0 bukan mangrove stabil, 1 mangrove stabil, 2 hilang ke air, 3 hilang ke daratan, 4 bertambah dari air, 5 bertambah dari daratan
CH[ok & ~A & ~B] = 0; CH[ok & A & B] = 1; CH[ok & A & ~B & W_B] = 2; CH[ok & A & ~B & ~W_B] = 3; CH[ok & ~A & B & W_A] = 4; CH[ok & ~A & B & ~W_A] = 5
NAMES = ['Bukan mangrove stabil', 'Mangrove stabil', 'Hilang ke air', 'Hilang ke daratan lain', 'Bertambah dari air', 'Bertambah dari daratan']
TRANS = pd.DataFrame([{'kelas': NAMES[i], 'luas_ha': round(float((CH == i).sum()) * ha, 1)} for i in range(1, 6)])
tot_l = TRANS.luas_ha[TRANS.kelas.str.startswith('Hilang')].sum(); tot_g = TRANS.luas_ha[TRANS.kelas.str.startswith('Bertambah')].sum()
TRANS['persen'] = [np.nan if k.startswith('Mangrove') else round(100 * v / max(tot_l if k.startswith('Hilang') else tot_g, 1e-9), 1) for k, v in zip(TRANS.kelas, TRANS.luas_ha)]
save_tab(TRANS, 'transisi_2005_2025', 'transitions_2005_2025'); print(TRANS.to_string(index=False))
np.savez_compressed(os.path.join(DIR_DATA, 'P04_peta_perubahan_2005_2025.npz'), X0=X0, Y0=Y0, RES=RES, crs=CRS, kelas=np.nan_to_num(CH, nan=255).astype(np.uint8))
CHK = []
if all(k in REF for k in ('GMW_2010', 'CGMD_2010', 'GMW_2020', 'CGMD_2020')):
    r10 = (REF['GMW_2010'] == 1) & (REF['CGMD_2010'] == 1); n10 = (REF['GMW_2010'] == 0) & (REF['CGMD_2010'] == 0)
    r20 = (REF['GMW_2020'] == 1) & (REF['CGMD_2020'] == 1); n20 = (REF['GMW_2020'] == 0) & (REF['CGMD_2020'] == 0)
    ok2 = np.isfinite(MAP[2010]) & np.isfinite(MAP[2020]); m10, m20 = MAP[2010] == 1, MAP[2020] == 1
    for nm, rm, det in [('Hilang (acuan sepakat)', r10 & n20, m10 & ~m20), ('Bertambah (acuan sepakat)', n10 & r20, ~m10 & m20),
                        ('Mangrove stabil (acuan sepakat)', r10 & r20, m10 & m20), ('Bukan mangrove stabil (acuan sepakat)', n10 & n20 & ZONE, ~m10 & ~m20)]:
        s = rm & ok2; CHK.append({'kelas': nm, 'n_piksel_acuan': int(s.sum()), 'terdeteksi_persen': round(100 * float(det[s].mean()), 1) if s.sum() else np.nan})
CHK = pd.DataFrame(CHK); save_tab(CHK, 'cek_perubahan_2010_2020', 'change_check_2010_2020'); print(CHK.to_string(index=False))

## Tahap 8 — Hubungan mangrove dengan abrasi (transek P01)
Data P01 dibaca dari Drive (`P01_abrasi/P01_hasil.zip`), atau Zenodo, atau GitHub. Untuk setiap transek P01 (setiap 100 m, 1.000 m ke darat dan 1.500 m ke laut dari garis acuan):
* **lebar mangrove** tahun e = panjang bagian transek yang mangrove (m);
* **tipe depan 2005**: *depan mangrove* bila ≥ 50% dari 150 m di belakang garis vegetasi 2005 adalah mangrove;
* dibandingkan dengan **LRR garis vegetasi 2005–2025** dari P01 (negatif = abrasi).

Uji: Mann-Whitney U dan Cliff's delta (depan mangrove vs bukan), korelasi Spearman. Selang kepercayaan 95% dengan **bootstrap blok** (blok 20 transek = 2 km) karena transek bertetangga tidak saling bebas.

In [ ]:
import requests
from scipy import stats
P01_DIR = os.path.join(BASE, 'P01_data'); os.makedirs(P01_DIR, exist_ok=True)
NEED = ['P01_transek_laju.csv', 'P01_posisi_garis_pantai.csv', 'P01_garis_pantai_transek.geojson']
def from_zip(zb, src):
    with zipfile.ZipFile(zb) as z:
        for f in NEED:
            mm = [n for n in z.namelist() if n.endswith(f)]
            if not mm: return False
            open(os.path.join(P01_DIR, f), 'wb').write(z.read(mm[0]))
    print('Data P01 dari', src); return True
def get_p01():
    if all(os.path.exists(os.path.join(P01_DIR, f)) for f in NEED): print('Data P01 sudah ada'); return True
    zp = os.path.join(DRIVE, 'P01_abrasi', 'P01_hasil.zip')
    if os.path.exists(zp) and from_zip(zp, 'Google Drive'): return True
    try:
        rec = requests.get(f'https://zenodo.org/api/records/{P01_DOI_RECORD}', timeout=60).json()
        for f in rec.get('files', []):
            url = f.get('links', {}).get('self') or f.get('links', {}).get('download')
            if url and f.get('key', '').endswith('.zip') and from_zip(io.BytesIO(requests.get(url, timeout=300).content), 'Zenodo'): return True
    except Exception as ex: print('Zenodo gagal:', str(ex)[:100])
    for br in ('main', 'master'):
        try:
            ok_ = True
            for f in NEED:
                r_ = requests.get(f'https://raw.githubusercontent.com/{P01_REPO}/{br}/data/{f}', timeout=120)
                if r_.status_code != 200: ok_ = False; break
                open(os.path.join(P01_DIR, f), 'wb').write(r_.content)
            if ok_: print('Data P01 dari GitHub'); return True
        except Exception as ex: print('GitHub gagal:', str(ex)[:100])
    return False
assert get_p01(), 'Data P01 tidak ditemukan. Upload P01_hasil.zip ke Google Drive folder P01_abrasi/ lalu jalankan ulang sel ini.'
TRL = pd.read_csv(os.path.join(P01_DIR, 'P01_transek_laju.csv')); POSP = pd.read_csv(os.path.join(P01_DIR, 'P01_posisi_garis_pantai.csv'))
GJ = json.load(open(os.path.join(P01_DIR, 'P01_garis_pantai_transek.geojson')))
TG = {f['properties']['transek']: f['geometry']['coordinates'] for f in GJ['features'] if 'transek' in f['properties']}
S = np.arange(-L_LAND, L_SEA + STEP, STEP)
tids = np.array(sorted(TG)); land = np.array([TG[t][0] for t in tids]); sea = np.array([TG[t][-1] for t in tids])
xl, yl = ll_to_utm(land[:, 0], land[:, 1]); xs_, ys_ = ll_to_utm(sea[:, 0], sea[:, 1])
LEN = np.hypot(np.array(xs_) - xl, np.array(ys_) - yl); print(f'{len(tids)} transek P01; panjang median {np.median(LEN):.0f} m (seharusnya {L_LAND + L_SEA} m)')
fr = (S + L_LAND) / (L_LAND + L_SEA)
PXc = (np.array(xl)[:, None] + fr[None] * (np.array(xs_) - xl)[:, None] - X0) / RES - 0.5
PYc = (Y0 - (np.array(yl)[:, None] + fr[None] * (np.array(ys_) - yl)[:, None])) / RES - 0.5
def prof(a): return ndi.map_coordinates(np.nan_to_num(a, nan=-1), [PYc.ravel(), PXc.ravel()], order=0, mode='constant', cval=-1).reshape(PXc.shape)
PROF = {e: prof(MAP[e]) for e in EPOCHS}
def width(p): v = p >= 0; return np.where(v.mean(1) > 0.8, (p == 1).sum(1) * STEP, np.nan)
veg05 = POSP[(POSP.garis == 'veg') & (POSP.tahun <= 2007)].sort_values('tahun').groupby('transek').posisi_m.first()
v05 = np.array([veg05.get(t, np.nan) for t in tids])
zone_f = (S[None] <= v05[:, None]) & (S[None] >= v05[:, None] - FRONT_M); p05 = PROF[2005]
with np.errstate(invalid='ignore'):
    ff = np.where(zone_f.sum(1) > 0, ((p05 == 1) & zone_f).sum(1) / np.maximum(((p05 >= 0) & zone_f).sum(1), 1), np.nan)
TT = pd.DataFrame({'transek': tids, 'lebar_2005_m': width(PROF[2005]), 'lebar_2025_m': width(PROF[2025]), 'posisi_veg_2005_m': v05, 'fraksi_depan_mangrove': ff})
for e in EPOCHS[1:-1]: TT[f'lebar_{e}_m'] = width(PROF[e])
TT['delta_lebar_m'] = TT.lebar_2025_m - TT.lebar_2005_m
TT = TT.merge(TRL[['transek', 'jarak_km', 'lon', 'lat', 'valid', 'LRR_veg', 'sig_veg', 'LRR_air']], on='transek', how='left')
TT['tipe_depan'] = np.where(TT.fraksi_depan_mangrove >= 0.5, 'Depan mangrove', np.where(TT.fraksi_depan_mangrove.notna(), 'Depan bukan mangrove', None))
TT.round(4).to_csv(os.path.join(DIR_DATA, 'P04_transek_mangrove.csv'), index=False)
A_ = TT[TT.valid.astype(bool) & TT.LRR_veg.notna() & TT.tipe_depan.notna() & TT.lebar_2005_m.notna()].sort_values('transek').reset_index(drop=True)
print(f'Transek dianalisis: {len(A_)} (depan mangrove {int((A_.tipe_depan == "Depan mangrove").sum())})')
def cliff(x, y):
    x, y = np.asarray(x), np.asarray(y)
    if len(x) == 0 or len(y) == 0: return np.nan
    return float((np.sign(x[:, None] - y[None]).sum()) / (len(x) * len(y)))
nb_ = int(np.ceil(len(A_) / BLK_T)); bid = np.arange(len(A_)) // BLK_T; brng = np.random.default_rng(SEED)
BS_IDX = [np.concatenate([np.flatnonzero(bid == b) for b in brng.integers(0, nb_, nb_)]) for _ in range(BOOT)]
def boot(fn):
    v = np.array([fn(A_.iloc[i]) for i in BS_IDX], float); v = v[np.isfinite(v)]
    return (np.percentile(v, 2.5), np.percentile(v, 97.5)) if len(v) > 50 else (np.nan, np.nan)
GRP_T = []
for nm, s in [('Depan mangrove', A_.tipe_depan == 'Depan mangrove'), ('Depan bukan mangrove', A_.tipe_depan == 'Depan bukan mangrove'), ('Semua', A_.tipe_depan.notna())]:
    g = A_[s]
    GRP_T.append({'tipe_depan': nm, 'n_transek': len(g), 'median_LRR_veg': g.LRR_veg.median(), 'abrasi_signifikan_persen': 100 * float((g.sig_veg.astype(bool) & (g.LRR_veg < 0)).mean()) if len(g) else np.nan,
                  'median_lebar_2005_m': g.lebar_2005_m.median(), 'median_lebar_2025_m': g.lebar_2025_m.median()})
GRP_T = pd.DataFrame(GRP_T).round(3); save_tab(GRP_T, 'abrasi_per_tipe_depan', 'erosion_by_front_type'); print(GRP_T.to_string(index=False))
xm, xn = A_.LRR_veg[A_.tipe_depan == 'Depan mangrove'], A_.LRR_veg[A_.tipe_depan == 'Depan bukan mangrove']
def mdiff(d): return d.LRR_veg[d.tipe_depan == 'Depan mangrove'].median() - d.LRR_veg[d.tipe_depan == 'Depan bukan mangrove'].median()
def cd(d): return cliff(d.LRR_veg[d.tipe_depan == 'Depan mangrove'], d.LRR_veg[d.tipe_depan == 'Depan bukan mangrove'])
TEST = []
if len(xm) > 2 and len(xn) > 2:
    U, pU = stats.mannwhitneyu(xm, xn, alternative='two-sided')
    TEST.append({'uji': 'Mann-Whitney U', 'statistik': U, 'nilai_p': pU, 'CI95_bawah': np.nan, 'CI95_atas': np.nan})
    TEST.append({'uji': 'Selisih median LRR_veg (mangrove − bukan)', 'statistik': mdiff(A_), 'nilai_p': np.nan, **dict(zip(['CI95_bawah', 'CI95_atas'], boot(mdiff)))})
    TEST.append({'uji': 'Cliff delta', 'statistik': cd(A_), 'nilai_p': np.nan, **dict(zip(['CI95_bawah', 'CI95_atas'], boot(cd)))})
TEST = pd.DataFrame(TEST); save_tab(TEST, 'uji_kelompok', 'group_tests'); print(TEST.round(4).to_string(index=False))
COR = []
for x, y in [('lebar_2005_m', 'LRR_veg'), ('lebar_2005_m', 'LRR_air'), ('delta_lebar_m', 'LRR_veg')]:
    d = A_[[x, y]].dropna()
    if len(d) < 10: continue
    rho, p = stats.spearmanr(d[x], d[y]); lo, hi = boot(lambda q, x=x, y=y: stats.spearmanr(q[x], q[y], nan_policy='omit')[0])
    COR.append({'variabel_x': x, 'variabel_y': y, 'n': len(d), 'rho_Spearman': rho, 'nilai_p': p, 'CI95_bawah': lo, 'CI95_atas': hi})
COR = pd.DataFrame(COR); save_tab(COR.round(4), 'korelasi', 'correlations'); print(COR.round(4).to_string(index=False))

## Tahap 9 — Gambar dua bahasa

In [ ]:
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap; from matplotlib.patches import Patch
plt.rcParams.update({'font.size': 9})
T = {'ID': dict(m='Mangrove', e='Epoch', ha='Luas mangrove (ha)', this='Peta ini', names=NAMES, km='Jarak sepanjang pantai (km)', lrr='LRR garis vegetasi (m/thn)',
                w='Lebar mangrove (m)', w05='Lebar mangrove 2005 (m)', fronts=['Depan mangrove', 'Depan bukan mangrove'], ch=f'Perubahan {EPOCHS[0]}–{EPOCHS[-1]}', nod='Tidak ada data'),
     'EN': dict(m='Mangrove', e='Epoch', ha='Mangrove area (ha)', this='This map', names=[VAL_EN[n] for n in NAMES], km='Alongshore distance (km)', lrr='Vegetation-line LRR (m/yr)',
                w='Mangrove width (m)', w05='Mangrove width 2005 (m)', fronts=['Mangrove-fronted', 'Non-mangrove-fronted'], ch=f'Change {EPOCHS[0]}–{EPOCHS[-1]}', nod='No data')}
CCOL = ['#f0f0f0', '#1b7837', '#2166ac', '#d6604d', '#80cdc1', '#fdb863']
ext = [0, W * RES / 1000, 0, H * RES / 1000]
for lang, t in T.items():
    fig, axs = plt.subplots(1, 3, figsize=(15, 5.2))
    for ax, e in zip(axs[:2], (EPOCHS[0], EPOCHS[-1])):
        ax.imshow(np.where(np.isfinite(MAP[e]), MAP[e], np.nan), cmap=ListedColormap(['#f0f0f0', '#1b7837']), vmin=0, vmax=1, extent=ext, interpolation='nearest')
        ax.set_title(f'{t["m"]} {e}'); ax.set_xlabel('km'); ax.set_ylabel('km')
    axs[2].imshow(CH, cmap=ListedColormap(CCOL), vmin=-0.5, vmax=5.5, extent=ext, interpolation='nearest'); axs[2].set_title(t['ch']); axs[2].set_xlabel('km')
    axs[2].legend(handles=[Patch(color=c, label=n) for c, n in zip(CCOL[1:], t['names'][1:])], loc='lower left', fontsize=7)
    fig.tight_layout(); fig.savefig(os.path.join(DIR_FIG, f'P04_gambar_peta_mangrove_{lang}.png'), dpi=250); plt.close(fig)
    fig, ax = plt.subplots(figsize=(6.5, 4))
    a_ = AREA[AREA.sumber == 'Peta ini']; ax.plot(a_.tahun, a_.luas_ha, 'o-', color='k', label=t['this'])
    for src, mk, c in [('GMW', 's', '#2166ac'), ('CGMD', '^', '#d6604d'), ('WorldCover', 'D', '#4daf4a')]:
        r_ = AREA[AREA.sumber == src]
        if len(r_): ax.plot(r_.tahun, r_.luas_ha, mk, ls='--', color=c, label=src)
    ax.set_xticks(range(2005, 2026, 5)); ax.set_xlabel(t['e']); ax.set_ylabel(t['ha']); ax.legend(fontsize=8); ax.grid(alpha=0.3)
    fig.tight_layout(); fig.savefig(os.path.join(DIR_FIG, f'P04_gambar_luas_mangrove_{lang}.png'), dpi=300); plt.close(fig)
    fig, axs = plt.subplots(1, 2, figsize=(11, 4.2))
    data_ = [A_.LRR_veg[A_.tipe_depan == k].values for k in ('Depan mangrove', 'Depan bukan mangrove')]
    axs[0].boxplot(data_, showfliers=False); axs[0].set_xticks([1, 2]); axs[0].set_xticklabels([f'{n}\n(n = {len(d)})' for n, d in zip(t['fronts'], data_)])
    for i, d in enumerate(data_): axs[0].plot(np.random.default_rng(i).normal(i + 1, 0.06, len(d)), d, '.', alpha=0.25, ms=3, color='#555')
    axs[0].axhline(0, color='k', lw=0.6); axs[0].set_ylabel(t['lrr'])
    axs[1].scatter(A_.lebar_2005_m, A_.LRR_veg, s=6, alpha=0.4, color='#1b7837'); axs[1].axhline(0, color='k', lw=0.6)
    if len(COR): r0 = COR.iloc[0]; axs[1].set_title(f'ρ = {r0.rho_Spearman:.2f} [{r0.CI95_bawah:.2f}, {r0.CI95_atas:.2f}]')
    axs[1].set_xlabel(t['w05']); axs[1].set_ylabel(t['lrr'])
    fig.tight_layout(); fig.savefig(os.path.join(DIR_FIG, f'P04_gambar_mangrove_vs_abrasi_{lang}.png'), dpi=300); plt.close(fig)
    fig, ax = plt.subplots(figsize=(11, 4)); B_ = TT[TT.valid.astype(bool)].sort_values('jarak_km')
    ax.bar(B_.jarak_km, B_.LRR_veg, width=0.1, color=np.where(B_.LRR_veg < 0, '#d6604d', '#4393c3')); ax.set_xlabel(t['km']); ax.set_ylabel(t['lrr']); ax.axhline(0, color='k', lw=0.5)
    ax2 = ax.twinx(); ax2.plot(B_.jarak_km, B_.lebar_2005_m, color='#1b7837', lw=1, label=f'{t["w"]} 2005'); ax2.plot(B_.jarak_km, B_.lebar_2025_m, color='#1b7837', lw=1, ls=':', label=f'{t["w"]} 2025')
    ax2.set_ylabel(t['w']); ax2.legend(loc='upper right', fontsize=8)
    fig.tight_layout(); fig.savefig(os.path.join(DIR_FIG, f'P04_gambar_sepanjang_pantai_{lang}.png'), dpi=300); plt.close(fig)
print(sorted(os.listdir(DIR_FIG)))

In [ ]:
# Ringkasan, Excel, zip
summary = {'bbox': BBOX, 'epoch': EPOCHS, 'citra_per_epoch': N_IMG, 'ambang_air': THR_W, 'peta_acuan': sorted(REF), 'n_titik_latih': {'mangrove': len(I_M), 'daratan': len(I_L), 'air': len(I_W)},
           'validasi': CV.to_dict('records'), 'luas': AREA.to_dict('records'), 'kesesuaian': AGR.to_dict('records'), 'perubahan_periode': PER.to_dict('records'),
           'transisi': TRANS.to_dict('records'), 'cek_perubahan': CHK.to_dict('records'), 'kelompok_transek': GRP_T.to_dict('records'), 'uji': TEST.to_dict('records'),
           'korelasi': COR.to_dict('records'), 'sumber_P01': f'https://doi.org/10.5281/zenodo.{P01_DOI_RECORD}'}
json.dump(summary, open(os.path.join(DIR_TAB, 'P04_hasil_ringkas.json'), 'w'), indent=2, ensure_ascii=False, default=lambda o: o.item() if hasattr(o, 'item') else str(o))
with pd.ExcelWriter(os.path.join(DIR_TAB, 'P04_semua_tabel.xlsx')) as xw:
    for f in sorted(glob.glob(os.path.join(DIR_TAB, '*.csv'))): pd.read_csv(f).to_excel(xw, sheet_name=os.path.basename(f)[4:-4][:31], index=False)
zpath = '/content/P04_hasil.zip' if os.path.exists('/content') else 'P04_hasil.zip'
with zipfile.ZipFile(zpath, 'w', zipfile.ZIP_DEFLATED) as z:
    for root, _, fs_ in os.walk(OUT):
        for fn in fs_: z.write(os.path.join(root, fn), os.path.relpath(os.path.join(root, fn), os.path.dirname(OUT)))
import shutil; shutil.copy(zpath, os.path.join(BASE, 'P04_hasil.zip'))
print(PER.to_string(index=False)); print(GRP_T.to_string(index=False))
try:
    from google.colab import files; files.download(zpath)
except Exception:
    print('Arsip:', zpath)
print('Salinan juga tersimpan di Drive:', os.path.join(BASE, 'P04_hasil.zip'))

Selesai. Upload `P04_hasil.zip` ke Claude. Salinannya juga ada di Google Drive `P04_mangrove/P04_hasil.zip`.